# Просмотр speech-данных
JupyterHub на dev-1; общий пакет `lalm.annotation` находится в репозитории IFAO-lalm.
Зависимости ядра: boto3, duckdb, pandas, IPython. Используйте read-only S3 credentials
через стандартные AWS-переменные окружения (или AWS profile), `S3_ENDPOINT` и
`AWS_CA_BUNDLE`. Не вставляйте ключи в notebook. Данные читаются из существующего MinIO.

Снимок предназначен для инспекции, не является обучающим релизом.
`not_linked_or_not_available` означает, что исходная метка пока не привязана или недоступна.
Ответ Gemini не означает ручную проверку. После работы очистите outputs перед сохранением в Git.


In [ ]:
import json, hashlib
from pathlib import Path
from IPython.display import Audio, display
from lalm.annotation.inspection import open_snapshot
s3, snapshot, cache, db = open_snapshot()
print('Снимок:', snapshot['snapshot_at'])
print({name: table['rows'] for name, table in snapshot['tables'].items()})


## Сводка и исходники, включая ещё не обработанные

In [ ]:
display(db.sql("SELECT corpus, preparation_state, count(*) AS files, sum(bytes) AS bytes FROM sources GROUP BY ALL ORDER BY corpus, preparation_state").df())
display(db.sql("SELECT * FROM progress ORDER BY corpus, prompt_variant, processing_state").df())


In [ ]:
corpus_filter = None  # Например: имя корпуса из сводки.
display(db.execute("SELECT * FROM sources WHERE (? IS NULL OR corpus=?) ORDER BY object_key LIMIT 50", [corpus_filter, corpus_filter]).df())


## Сравнение исходной разметки и проходов Gemini
Режим `flagged` показывает **50 наиболее проблемных примеров**, а не случайную выборку качества.
`attached` означает только наличие исходной метки; её правильность не подтверждена.
`None` — отсутствие принятого текста: смотрите `error_codes` и статусы отдельных проходов.
`pilot_review_only` — результат ожидает проверки, а не технический сбой. `native_result_rejected` — ответ не прошёл проверку протокола.


In [ ]:
from lalm.annotation.inspection import agreement_summary
import pandas as pd
all_examples = db.execute("SELECT * FROM examples WHERE (? IS NULL OR corpus=?) ORDER BY updated_at DESC", [corpus_filter, corpus_filter]).df()
agreement = pd.DataFrame([agreement_summary(row) for row in all_examples.to_dict('records')], index=all_examples.index, columns=['agreement_status', 'agreement_passes', 'three_pass_agreement', 'source_conditioned_agreement'])
all_examples = pd.concat([all_examples, agreement], axis=1)
display(all_examples.groupby('agreement_status').size().rename('examples').to_frame())
selection_mode = 'matched'  # matched / three_matched / flagged / latest
if selection_mode == 'matched':
    selected_rows = all_examples[all_examples['agreement_status'].isin(['2 совпали', '3 совпали'])]
elif selection_mode == 'three_matched':
    selected_rows = all_examples[all_examples['three_pass_agreement'] == True]
elif selection_mode == 'flagged':
    selected_rows = all_examples.sort_values(['flagged_passes', 'updated_at'], ascending=False)
else:
    selected_rows = all_examples
examples = selected_rows.head(50).reset_index(drop=True)
print('Режим:', selection_mode, 'Найдено:', len(selected_rows), 'Показано:', len(examples))
print('Совпадение после нормализации регистра/пунктуации/ё; это не human-gold.')
columns = ['sample_id', 'corpus', 'agreement_status', 'source_conditioned_agreement', 'original_annotation_json', 'gemini_verbatim', 'gemini_coverage', 'gemini_literal_audit', 'gemini_source_correction', 'detected_languages', 'error_codes', 'passes_requested', 'passes_with_text', 'unfinished_passes']
display(examples[[c for c in columns if c in examples.columns]])


In [ ]:
# Выберите номер строки таблицы выше.
example_index = 0
if examples.empty:
    print('Примеры пока отсутствуют для выбранного корпуса.')
else:
    selected = examples.iloc[example_index]
    passes = db.execute("SELECT * FROM passes WHERE batch_id IS NOT DISTINCT FROM ? AND sample_id=? ORDER BY prompt_variant", [selected['batch_id'], selected['sample_id']]).df()
    display(passes[['prompt_variant', 'processing_state', 'error_code', 'original_annotation', 'gemini_text', 'language', 'model', 'contract_sha256']])
    audio = s3.get_object(Bucket=selected['audio_bucket'], Key=selected['audio_key'])['Body'].read()
    assert hashlib.sha256(audio).hexdigest() == selected['audio_sha256']
    display(Audio(audio))
    provenance = json.loads(passes.iloc[0]['source_provenance'] or '{}')
    print('Исходник и границы фрагмента:')
    display(provenance)


## Исходный объект
Исходник может быть аудио, большим архивом или Parquet с вложенным аудио.
`source_provenance` показывает его точное расположение и элемент (`member`).
Следующая функция выводит метаданные без скачивания многогигабайтного архива.
Для самостоятельного аудиофайла доступно прослушивание (лимит 50 MiB).
Для архивов/Parquet выше прослушивается подготовленный фрагмент, не весь исходный контейнер.


In [ ]:
def inspect_source(source_bucket, source_key, play=False):
    meta = s3.head_object(Bucket=source_bucket, Key=source_key)
    print({'uri': f's3://{source_bucket}/{source_key}', 'bytes': meta['ContentLength'], 'etag': meta['ETag']})
    if play:
        if Path(source_key).suffix.lower() not in {'.wav', '.mp3', '.opus', '.ogg', '.flac', '.m4a'}:
            raise ValueError('Это контейнер данных; выберите его аудиофрагмент через examples.')
        if meta['ContentLength'] > 50 * 1024**2:
            raise ValueError('Слишком большой файл для inline-плеера; используйте подготовленный фрагмент.')
        display(Audio(s3.get_object(Bucket=source_bucket, Key=source_key)['Body'].read()))

# Можно указать bucket/object_key любой строки sources, в том числе необработанной.
# inspect_source('speech-corpora', 'staging/...', play=False)
if not examples.empty and provenance.get('bucket') and provenance.get('key'):
    inspect_source(provenance['bucket'], provenance['key'])


## Опубликованные результаты
Пустая таблица означает отсутствие опубликованного релиза, а не потерю исходников или промежуточных ответов.

In [ ]:
display(db.sql('SELECT * FROM publications ORDER BY created_at DESC').df())


## Входные промпты и исходные ответы
Не сохраняйте тексты production-примеров в Git.


In [ ]:
# Все сохранённые входы/выходы выбранных запросов: prompt, request JSON, raw response.
# Сначала только список объектов; содержимое открывайте выборочно.
if not examples.empty:
    for prefix in passes['raw_result_prefix'].dropna().unique():
        objects = s3.list_objects_v2(Bucket='speech-annotations', Prefix=prefix, MaxKeys=30)
        display([{'key': x['Key'], 'bytes': x['Size']} for x in objects.get('Contents', [])])

# from lalm.annotation.inspection import read_text_object
# print(read_text_object(s3, 'speech-annotations', 'ключ из списка выше'))
